# PII Masking: DeBERTa Fine-Tuning and LLaMA Zero-Shot

1. Inspect the provided NER data.
2. Fine-tune DeBERTa for token classification over `O`, `B-PER`, `I-PER`, and `B-EMAIL`.
3. Evaluate on the test set with accuracy, precision, recall, F1, FPR, and FNR.
4. Run LLaMA 1B as a zero-shot baseline through prompting and JSON parsing.
5. Compare both approaches and analyze results.

Model IDs:

- DeBERTa: `microsoft/deberta-v3-base`
- LLaMA 1B: `meta-llama/Llama-3.2-1B-Instruct`


### DeBERTa 

In [ ]:
import os
import json
import numpy as np
from collections import Counter
import evaluate
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix


In [ ]:
BASE_MODEL_DIR = r'D:/Coding/Models/DeBERTa-v3-base'
CHECKPOINT_DIR = r'D:/Coding/Models/DeBERTa-v3-pii-masking'
FINAL_MODEL_DIR = r'D:/Coding/Models/DeBERTa-v3-pii-masking-final'
LOG_DIR = r'D:/Coding/Models/Logs'
TOKENIZER_LOAD_PATH = BASE_MODEL_DIR

if os.path.isdir(FINAL_MODEL_DIR):           # Load final model if it exists else use the base model
    MODEL_LOAD_PATH = FINAL_MODEL_DIR
    TRAIN = False
    SAVE_FINAL = False
    print(f'Loading final trained model from {FINAL_MODEL_DIR}')

else:
    MODEL_LOAD_PATH = BASE_MODEL_DIR
    TRAIN = True
    SAVE_FINAL = True
    print('No final trained model found. Training from the base model.')

SEED = 42
MAX_LENGTH = 512
VALIDATION_SIZE = 0.15
ENTITY_TYPES = ['EMAIL', 'PER']
USE_CUDA = torch.cuda.is_available()
USE_BF16 = USE_CUDA and torch.cuda.is_bf16_supported()

if USE_CUDA:                                       # To leverage the use of GPU if found
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    print('CUDA is not available.')


def load_data():
    with open('../datasets/train_data_modified.json', 'r', encoding='utf-8') as file:
        train_data = json.load(file)
    
    with open('../datasets/test_data_modified.json', 'r', encoding='utf-8') as file:
        test_data = json.load(file)

    print(f'Loaded training set with {len(train_data)} lines and testing set with {len(test_data)} lines.')
    return train_data, test_data

train_data, test_data = load_data()


def validate_records(data, label):
    counts = {}
    for entry in data:
        tokens = entry['tokens']
        tags = entry['ner_tags']

        if len(tags) != len(tokens): 
            raise ValueError(f'Length mismatch between tags and tokens.')
        
        if not isinstance(tags, list) and not isinstance(tokens, list):
            raise ValueError(f'Tags and tokens are not lists. Both must be lists.')

        for tag in tags:
            if tag not in counts:
                counts[tag] = 0
            counts[tag] += 1

    print(f'{label} label counts: {counts}')
    return counts


train_tag_counts = validate_records(train_data, 'Training set')
test_tag_counts = validate_records(test_data, 'Testing set')

label_list = ['O', 'B-PER', 'I-PER', 'B-EMAIL', 'I-EMAIL']
label2id = {label: idx for idx, label in enumerate(label_list)}
id2label = {idx: label for label, idx in label2id.items()}

full_dataset = Dataset.from_list(train_data)
split_dataset = full_dataset.train_test_split(test_size=VALIDATION_SIZE, seed=SEED)
train_ds = split_dataset['train']
validation_ds = split_dataset['test']
test_ds = Dataset.from_list(test_data)

print(f'Train split     : {len(train_ds)}')
print(f'Validation split: {len(validation_ds)}')
print(f'Test split      : {len(test_ds)}')

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_LOAD_PATH, fix_mistral_regex=True)


def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        max_length=MAX_LENGTH,
        is_split_into_words=True
    )

    labels = []
    for i, word_labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        prev = None
        label_ids = []

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)

            elif word_id != prev:
                label_ids.append(label2id[word_labels[word_id]])

            else:
                label = word_labels[word_id]
                if label.startswith("B-"):
                    label = "I-" + label[2:]

                label_ids.append(label2id[label])

            prev = word_id

        labels.append(label_ids)

    tokenized["labels"] = labels
    return tokenized


train_tok = train_ds.map(tokenize_and_align_labels, batched=True, remove_columns=train_ds.column_names)

val_tok = validation_ds.map(tokenize_and_align_labels, batched=True, remove_columns=validation_ds.column_names)

test_tok = test_ds.map(tokenize_and_align_labels, batched=True, remove_columns=test_ds.column_names)


def count_aligned_labels(dataset):
    counts = np.zeros(len(label_list), dtype=np.int64)
    for row in dataset:
        labels = np.array(row['labels'])
        labels = labels[labels != -100]
        counts += np.bincount(labels, minlength=len(label_list))
    return counts


aligned_counts = count_aligned_labels(train_tok)
print('Aligned train label counts:', {label: int(aligned_counts[idx]) for idx, label in enumerate(label_list)})

model = AutoModelForTokenClassification.from_pretrained(
        MODEL_LOAD_PATH,
        num_labels=len(label_list),
        id2label=id2label,
        label2id=label2id,
        dtype=torch.float32
)
    

model.float()
if not all(torch.isfinite(param).all().item() for param in model.parameters()):
    raise ValueError('Base model contains NaN/Inf weights before training.')

metric = evaluate.load('seqeval')


def metric_inputs_from_logits(logits, labels):
    predictions = np.argmax(logits, axis=-1)
    true_predictions = []
    true_labels = []

    for pred_row, label_row in zip(predictions, labels):
        pred_tags = []
        label_tags = []
        for pred_id, label_id in zip(pred_row, label_row):
            if label_id == -100:
                continue
            pred_tags.append(label_list[int(pred_id)])
            label_tags.append(label_list[int(label_id)])
        true_predictions.append(pred_tags)
        true_labels.append(label_tags)

    return true_predictions, true_labels


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    true_predictions, true_labels = metric_inputs_from_logits(logits, labels)
    results = metric.compute(predictions=true_predictions, references=true_labels, zero_division=0)
    return {
        'precision': results['overall_precision'],
        'recall': results['overall_recall'],
        'f1': results['overall_f1'],
        'accuracy': results['overall_accuracy']
    }


class WeightedTokenTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get('labels')
        outputs = model(**inputs)
        logits = outputs.get('logits')
        loss_fct = torch.nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device),
            ignore_index=-100
        )
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


safe_counts = np.maximum(aligned_counts, 1)
weights = np.sqrt(safe_counts.sum() / (len(label_list) * safe_counts))
weights = np.clip(weights, 0.25, 6.0)
class_weights = torch.tensor(weights, dtype=torch.float32)
print('Loss weights:', {label: round(float(class_weights[idx]), 3) for idx, label in enumerate(label_list)})

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    logging_dir=LOG_DIR,
    seed=SEED,
    data_seed=SEED,
    fp16=False,
    bf16=USE_BF16,
    tf32=USE_CUDA,
    num_train_epochs=5,
    learning_rate=1e-5,
    warmup_steps=0.06,
    weight_decay=0.01,
    max_grad_norm=1.0,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    report_to='none'
)

trainer = WeightedTokenTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics
)

if TRAIN:
    trainer.train()
else:
    print('Skipping training, using the saved trained model.')

if SAVE_FINAL:
    trainer.save_model(FINAL_MODEL_DIR)
    tokenizer.save_pretrained(FINAL_MODEL_DIR)
    print(f'Saved final model and tokenizer to {FINAL_MODEL_DIR}')


def entity_type(tag):
    if tag == 'O' or '-' not in tag:
        return 'O'
    return tag.split('-', 1)[1]


def safe_divide(numerator, denominator):
    return numerator / denominator if denominator else 0.0


def print_report_row(name, precision, recall, f1, support):
    print(f'{name:>12} {precision:>10.2f} {recall:>9.2f} {f1:>9.2f} {support:>9}')


def print_detailed_evaluation(title, dataset):
    output = trainer.predict(dataset)
    predictions, references = metric_inputs_from_logits(output.predictions, output.label_ids)
    results = metric.compute(predictions=predictions, references=references, zero_division=0)

    entity_results = {
        entity: results.get(entity, {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 0})
        for entity in ENTITY_TYPES
    }
    total_support = int(sum(entity_results[entity]['number'] for entity in ENTITY_TYPES))

    macro_precision = safe_divide(sum(entity_results[e]['precision'] for e in ENTITY_TYPES), len(ENTITY_TYPES))
    macro_recall = safe_divide(sum(entity_results[e]['recall'] for e in ENTITY_TYPES), len(ENTITY_TYPES))
    macro_f1 = safe_divide(sum(entity_results[e]['f1'] for e in ENTITY_TYPES), len(ENTITY_TYPES))

    weighted_precision = safe_divide(
        sum(entity_results[e]['precision'] * entity_results[e]['number'] for e in ENTITY_TYPES),
        total_support
    )
    weighted_recall = safe_divide(
        sum(entity_results[e]['recall'] * entity_results[e]['number'] for e in ENTITY_TYPES),
        total_support
    )
    weighted_f1 = safe_divide(
        sum(entity_results[e]['f1'] * entity_results[e]['number'] for e in ENTITY_TYPES),
        total_support
    )

    print('\n' + '=' * 60)
    print(f'DETAILED EVALUATION: {title}')
    print('=' * 60)
    print(f"{'':>12} {'precision':>10} {'recall':>9} {'f1-score':>9} {'support':>9}\n")

    for entity in ENTITY_TYPES:
        row = entity_results[entity]
        print_report_row(entity, row['precision'], row['recall'], row['f1'], int(row['number']))

    print()
    print_report_row('micro avg', results['overall_precision'], results['overall_recall'], results['overall_f1'], total_support)
    print_report_row('macro avg', macro_precision, macro_recall, macro_f1, total_support)
    print_report_row('weighted avg', weighted_precision, weighted_recall, weighted_f1, total_support)
    print()

    flat_predictions = [tag for row in predictions for tag in row]
    flat_references = [tag for row in references for tag in row]

    for entity in ['PER', 'EMAIL']:
        true_positive = false_positive = true_negative = false_negative = 0
        for predicted_tag, reference_tag in zip(flat_predictions, flat_references):
            predicted_match = entity_type(predicted_tag) == entity
            reference_match = entity_type(reference_tag) == entity

            if predicted_match and reference_match:
                true_positive += 1
            elif predicted_match and not reference_match:
                false_positive += 1
            elif not predicted_match and reference_match:
                false_negative += 1
            else:
                true_negative += 1

        precision = safe_divide(true_positive, true_positive + false_positive)
        recall = safe_divide(true_positive, true_positive + false_negative)
        f1 = safe_divide(2 * precision * recall, precision + recall)
        fpr = safe_divide(false_positive, false_positive + true_negative)
        fnr = safe_divide(false_negative, false_negative + true_positive)

        print(f'{entity}: Security Metrics:')
        print(f'  Precision : {precision:.4f}')
        print(f'  Recall    : {recall:.4f}')
        print(f'  F1        : {f1:.4f}')
        print(f'  FPR       : {fpr:.4f}  (aggressive redaction risk)')
        print(f'  FNR       : {fnr:.4f}  (unmasked PII / data breach risk)\n')

    return results


validation_results = print_detailed_evaluation('VALIDATION SET', val_tok)
test_results = print_detailed_evaluation('TEST SET (WikiNeural)', test_tok)

Loading final trained model from D:/Coding/Models/DeBERTa-v3-pii-masking-final
Loaded training set with 28516 lines and testing set with 3650 lines.
Training set label counts: {'O': 658247, 'B-PER': 40270, 'I-PER': 29460, 'B-EMAIL': 15952}
Testing set label counts: {'O': 80427, 'B-PER': 5210, 'I-PER': 3956, 'B-EMAIL': 2157}
Train split     : 24238
Validation split: 4278
Test split      : 3650


Map:   0%|          | 0/24238 [00:00<?, ? examples/s]

Map:   0%|          | 0/4278 [00:00<?, ? examples/s]

Map:   0%|          | 0/3650 [00:00<?, ? examples/s]

Aligned train label counts: {'O': 688417, 'B-PER': 34213, 'I-PER': 82939, 'B-EMAIL': 13561, 'I-EMAIL': 109951}


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Loss weights: {'O': 0.52, 'B-PER': 2.33, 'I-PER': 1.497, 'B-EMAIL': 3.702, 'I-EMAIL': 1.3}
Skipping training, using the saved trained model.



DETAILED EVALUATION: VALIDATION SET
              precision    recall  f1-score   support

       EMAIL       1.00      1.00      1.00      2391
         PER       0.97      0.98      0.98      6057

   micro avg       0.98      0.99      0.98      8448
   macro avg       0.98      0.99      0.99      8448
weighted avg       0.98      0.99      0.98      8448

PER: Security Metrics:
  Precision : 0.9861
  Recall    : 0.9921
  F1        : 0.9891
  FPR       : 0.0020  (aggressive redaction risk)
  FNR       : 0.0079  (unmasked PII / data breach risk)

EMAIL: Security Metrics:
  Precision : 0.9998
  Recall    : 1.0000
  F1        : 0.9999
  FPR       : 0.0000  (aggressive redaction risk)
  FNR       : 0.0000  (unmasked PII / data breach risk)




DETAILED EVALUATION: TEST SET (WikiNeural)
              precision    recall  f1-score   support

       EMAIL       0.99      1.00      1.00      2157
         PER       0.98      0.98      0.98      5210

   micro avg       0.98      0.98      0.98      7367
   macro avg       0.99      0.99      0.99      7367
weighted avg       0.98      0.98      0.98      7367

PER: Security Metrics:
  Precision : 0.9894
  Recall    : 0.9894
  F1        : 0.9894
  FPR       : 0.0016  (aggressive redaction risk)
  FNR       : 0.0106  (unmasked PII / data breach risk)

EMAIL: Security Metrics:
  Precision : 0.9996
  Recall    : 0.9997
  F1        : 0.9997
  FPR       : 0.0001  (aggressive redaction risk)
  FNR       : 0.0003  (unmasked PII / data breach risk)

